In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/K2Debug/Financial-News-Sentiment-Analyzer-for-Economic-Forecasting.git"
PROJECT_ROOT = Path("/content/EF-02")
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        print("Workspace not found — cloning repo and installing dependencies...")
        !git clone {REPO_URL} {PROJECT_ROOT}
        !pip install -q python-dotenv groq openai bs4
        !pip install -q -r /content/EF-02/requirements.txt
        print("Workspace ready.")
    os.chdir(NOTEBOOKS_DIR)

# 02 - Model Benchmarking
**EF-02 | Tanzania Financial News Sentiment Analyser**

This notebook tests six sentiment models against a labelled synthetic dataset to determine which is most accurate for Tanzanian financial headlines.

Models tested in order: TextBlob, VADER, FinBERT, GPT-4o-mini, Llama 3.1 8b, Llama 3.3 70b

Dataset: `data/raw/project_testing.csv`

## 1. Imports and Setup

In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score

from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import pipeline
from openai import OpenAI

from groq import Groq

from pathlib import Path

# Prefer monorepo root ef02/.env; fall back to research/.env
load_dotenv(Path("..") / ".." / ".env")
load_dotenv(Path("..") / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY")


## 2. Load Dataset

In [2]:
df = pd.read_csv("../data/raw/project_testing.csv", on_bad_lines="skip")

df_results = df[["headline", "sentiment"]].copy()

print(f"Dataset size: {len(df)} headlines")
print(f"Label distribution:\n{df['sentiment'].value_counts()}")
df.head()

Dataset size: 504 headlines
Label distribution:
sentiment
Positive    247
Negative    164
Neutral      93
Name: count, dtype: int64


,date,headline,source,category,sentiment,tzs_usd_rate,inflation_rate,bot_policy_rate
0,2021-01-05,Bank of Tanzania holds policy rate steady amid...,Daily News,Policy,Neutral,2296,3.1,5.0
1,2021-01-08,Tanzanian shilling opens week firmer as dollar...,The Citizen,Forex,Positive,2291,3.1,5.0
2,2021-01-12,KCB Tanzania reports 18% rise in mobile loan d...,Business Daily Africa,Fintech,Positive,2294,3.1,5.0
3,2021-01-15,Tanzania Revenue Authority misses January targ...,The EastAfrican,Taxation,Negative,2297,3.2,5.0
4,2021-01-19,Cashew export volumes from Mtwara port fall sh...,Mwananchi,Agriculture,Negative,2300,3.2,5.0


## 3. TextBlob

Rule-based model using polarity scores. Thresholds: >0.05 Positive, <-0.05 Negative, else Neutral.

In [3]:
def textblob_sentiment(headline):
    score = TextBlob(headline).sentiment.polarity
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

df_results["textblob"] = df_results["headline"].apply(textblob_sentiment)

textblob_accuracy = accuracy_score(df_results["sentiment"], df_results["textblob"])
print(f"TextBlob accuracy: {textblob_accuracy:.2%}")

TextBlob accuracy: 28.17%


## 4. VADER

Rule-based model built for social media text. Uses compound score: >0.05 Positive, <-0.05 Negative, else Neutral.

In [4]:
# run once if you haven't downloaded the lexicon: nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

def vader_sentiment(headline):
    compound = sia.polarity_scores(headline)["compound"]
    if compound > 0.05:
        return "Positive"
    elif compound < -0.05:
        return "Negative"
    else:
        return "Neutral"

df_results["vader"] = df_results["headline"].apply(vader_sentiment)

vader_accuracy = accuracy_score(df_results["sentiment"], df_results["vader"])
print(f"VADER accuracy: {vader_accuracy:.2%}")

VADER accuracy: 50.00%


## 5. FinBERT

Transformer model fine-tuned on financial text. Picks the highest confidence label from the three outputs.

In [5]:
finbert = pipeline("text-classification", model="ProsusAI/finbert")

def finbert_sentiment(headline):
    result = finbert(headline)
    best = max(result, key=lambda x: x["score"])
    return best["label"].title()

df_results["finbert"] = df_results["headline"].apply(finbert_sentiment)

finbert_accuracy = accuracy_score(df_results["sentiment"], df_results["finbert"])
print(f"FinBERT accuracy: {finbert_accuracy:.2%}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT accuracy: 66.67%


## 6. LLM Models (GPT-4o-mini, Llama 3.1 8b, Llama 3.3 70b)

All three LLMs use the same system prompt and batch runner. Groq models are accessed via the OpenAI-compatible SDK pointed at Groq's base URL.

Classification is by implied economic outcome, not emotional tone.

In [6]:
import os
# clients

openai_client = OpenAI(api_key=OPENAI_API_KEY)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [7]:
SYSTEM_PROMPT = """
You are classifying the economic impact of East African financial news headlines.

Positive: likely improves growth, stability, investment, consumer conditions, or business activity.
Negative: indicates contraction, financial loss, increased risk, market decline, or unfavorable regulatory changes.
Neutral: informational updates, budget approvals, or announcements with no clear directional outcome.

Classify by the implied economic effect, not emotional tone.
Return only a numbered list matching the input. One label per line. No extra text.

Example output:
1. Positive
2. Neutral
3. Negative
"""

MODELS = {
    "gpt-4o-mini":            openai_client,
    "llama-3.1-8b-instant":   groq_client,
    "llama-3.3-70b-versatile": groq_client,
}

In [9]:
def llm_sentiment_batch(headlines, client, model_name, batch_size=25):
    """
    Sends headlines to an LLM in batches of 25.
    Validates output length and label validity.
    Retries up to 3 times on failure.
    """
    VALID_LABELS = {"Positive", "Negative", "Neutral"}
    all_results = []
    batches = [headlines[i:i+batch_size] for i in range(0, len(headlines), batch_size)]

    for batch_num, batch in enumerate(batches):
        numbered = "\n".join([f"{i+1}. {h}" for i, h in enumerate(batch)])
        batch_results = None

        for attempt in range(1, 4):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": numbered}
                    ],
                    temperature=0,
                    max_tokens=500
                )
                lines = response.choices[0].message.content.strip().split("\n")
                parsed = [l.split(". ", 1)[1].strip() for l in lines if ". " in l]

                if len(parsed) == len(batch) and set(parsed).issubset(VALID_LABELS):
                    batch_results = parsed
                    break
                else:
                    print(f"  Batch {batch_num+1} attempt {attempt}: validation failed, retrying...")
                    time.sleep(2)

            except Exception as e:
                print(f"  Batch {batch_num+1} attempt {attempt} error: {e}")
                time.sleep(3)

        if batch_results is None:
            print(f"  Batch {batch_num+1} failed after 3 attempts. Filling with None.")
            batch_results = [None] * len(batch)

        all_results.extend(batch_results)

        if batch_num < len(batches) - 1:
            time.sleep(1.5)

    return all_results

In [12]:
def llm_sentiment_batch(headlines, client, model_name, batch_size=25):
    """
    Sends headlines to an LLM in batches of 25.
    Validates output length and label validity.
    Retries up to 3 times on failure.
    """
    VALID_LABELS = {"Positive", "Negative", "Neutral"}
    all_results = []
    batches = [headlines[i:i+batch_size] for i in range(0, len(headlines), batch_size)]

    print(f"Preparing to send {len(batches)} batches...")

    for batch_num, batch in enumerate(batches):
        numbered = "\n".join([f"{i+1}. {h}" for i, h in enumerate(batch)])
        batch_results = None

        for attempt in range(1, 4):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": numbered}
                    ],
                    temperature=0,
                    max_tokens=500
                )
                lines = response.choices[0].message.content.strip().split("\n")
                parsed = [l.split(". ", 1)[1].strip() for l in lines if ". " in l]

                if len(parsed) == len(batch) and set(parsed).issubset(VALID_LABELS):
                    batch_results = parsed
                    break
                else:
                    print(f"  Batch {batch_num+1} attempt {attempt}: validation failed, retrying...")
                    time.sleep(2)

            except Exception as e:
                print(f"  Batch {batch_num+1} attempt {attempt} error: {e}")
                time.sleep(3)

        if batch_results is None:
            print(f"  Batch {batch_num+1} failed after 3 attempts. Filling with None.")
            batch_results = [None] * len(batch)

        all_results.extend(batch_results)

        print(f"Batch {batch_num+1} complete", end=", ")

        if batch_num < len(batches) - 1:
            time.sleep(1.5)

    print(f"\n\nSuccessfully processed {len(batches)} batches!")

    return all_results


In [9]:
headlines_list = df_results["headline"].tolist()

llm_results = {}

for model_name, client in MODELS.items():
    print(f"Running {model_name}...")
    llm_results[model_name] = llm_sentiment_batch(headlines_list, client, model_name)
    print(f"Done.\n")

for model_name, preds in llm_results.items():
    df_results[model_name] = preds

Running gpt-4o-mini...
Preparing to send 21 batches...
Batch 1 complete, Batch 2 complete, Batch 3 complete, Batch 4 complete, Batch 5 complete, Batch 6 complete, Batch 7 complete, Batch 8 complete, Batch 9 complete, Batch 10 complete, Batch 11 complete, Batch 12 complete, Batch 13 complete, Batch 14 complete, Batch 15 complete, Batch 16 complete, Batch 17 complete, Batch 18 complete, Batch 19 complete, Batch 20 complete, Batch 21 complete, 
Successfully created 21 batches!
Done.

Running llama-3.1-8b-instant...
Preparing to send 21 batches...
Batch 1 complete, Batch 2 complete, Batch 3 complete, Batch 4 complete, Batch 5 complete, Batch 6 complete, Batch 7 complete, Batch 8 complete, Batch 9 complete, Batch 10 complete, Batch 11 complete, Batch 12 complete, Batch 13 complete, Batch 14 complete, Batch 15 complete, Batch 16 complete, Batch 17 complete, Batch 18 complete, Batch 19 complete, Batch 20 complete, Batch 21 complete, 
Successfully created 21 batches!
Done.

Running llama-3.3-7

## Accuracy Summary

In [13]:
true_labels = df_results["sentiment"]

model_columns = {
    "TextBlob":              "textblob",
    "VADER":                 "vader",
    "FinBERT":               "finbert",
    "GPT-4o-mini":           "gpt-4o-mini",
    "Llama 3.1 8b":          "llama-3.1-8b-instant",
    "Llama 3.3 70b":         "llama-3.3-70b-versatile",
}

summary_rows = []
for model_name, col in model_columns.items():
    acc = accuracy_score(true_labels, df_results[col])
    summary_rows.append({"Model": model_name, "Accuracy": f"{acc:.2%}"})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

        Model Accuracy
     TextBlob   28.17%
        VADER   50.00%
      FinBERT   66.67%
  GPT-4o-mini   92.46%
 Llama 3.1 8b   92.46%
Llama 3.3 70b   92.66%


In [12]:
# save full results for reference
df_results.to_csv("../data/processed/benchmarking_results.csv", index=False)
print("Results saved to data/processed/benchmarking_results.csv")

Results saved to data/processed/benchmarking_results.csv
